In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
gender = pd.read_csv('data-pemilih-kpu.csv')

In [ ]:
gender.info()

In [ ]:
gender['jenis_kelamin'].value_counts()

In [ ]:
gender['jenis_kelamin'] = gender['jenis_kelamin'].map({'Laki-Laki': 0, 'Perempuan': 1})

## **MODEL**

### Independent & Dependent

In [ ]:
X = gender.drop('jenis_kelamin', axis=1)
y = gender['jenis_kelamin']

### Feature Scaling

In [ ]:
# # Normalisasi
# X = (X - X.min()) / (X.max() - X.min())

# # Standarisasi
# X = (X - X.mean()) / X - X.std()

### Split Train Test

In [ ]:
np.random.seed(42)

idx_0 = np.where(y == 0)[0]
idx_1 = np.where(y == 1)[0]

np.random.shuffle(idx_0)
np.random.shuffle(idx_1)

train_0 = int(len(idx_0) * 0.8)
train_1 = int(len(idx_1) * 0.8)

train_idx = np.concatenate([idx_0[:train_0], idx_1[:train_1]])
test_idx = np.concatenate([idx_0[train_0:], idx_1[train_1:]])

np.random.shuffle(train_idx)
np.random.shuffle(test_idx)

X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]
X_test = X.iloc[test_idx]
y_test = y.iloc[test_idx]

### Resampling

In [ ]:
np.random.seed(42)

data_major = gender[gender['jenis_kelamin'] == 1]
data_minor = gender[gender['jenis_kelamin'] == 0]

n_major = len(data_major)
n_minor = len(data_minor)

idx_minor = data_minor.index

idx_random = np.random.choice(idx_minor, size=n_major, replace=True)
data_minor_over = gender.loc[idx_random]
gender = pd.concat([data_major, data_minor_over])

## **MODEL**

In [ ]:
def linear(X, w, b):
    z = np.dot(X - w) + b
    return z

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
def log_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-15, 1 -1e-15)
    loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return loss

In [ ]:
def gradient(X, y_true, y_pred):
    m = X.shape[0]
    error = y_pred - y_true

    dw = (1 / m) * np.dot(X.T, error)
    db = (1 / m) * np.sum(error)
    return dw, db

In [ ]:
def update_parameter(w, b, dw, db, learning_rate):
    w_new = w - learning_rate * dw
    b_new = b - learning_rate * db

    return w_new, b_new

In [ ]:
# Insialisasi Variabel
w = np.zeros(X_train.shape[1])
b = 0
learning_rate = 5
ephocs = 500
loss_history = []

# Looping 
for i in range(ephocs):
    z = linear(X_train, w, b)
    y_pred = sigmoid(z)

    loss_values = log_loss(y_train, y_pred)
    loss_history.apend(loss_values)

    dw, db = gradient(X_train, y_train, y_pred)
    w, b = update_parameter(w, b, dw, db, learning_rate)

    print(f"ephoch ke: {i}, loss: {loss_values:.4f}")

print(f"\nLoss akhir: {loss_values}")

### **EVLUASI**

### Fungsi Prediksi

In [2]:
def prediksi(X, w, b, threshold=0.5):
    z = linear(X, w, b)
    y_prob = sigmoid(z)
    y_class = [1 if p > threshold else 0 for p in y_prob]

    return np.array(y_class)

### Fungsi Evaluasi

In [3]:
def evaluate(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return accuracy, precision, recall, f1

### Prediksi Train

In [4]:
prediksi_train = prediksi(X_train, w, b)
akruasi_train = np.sum(prediksi_train == y_train) / len(X_train)
print(f"Prediksi Train\nAkurasi Train: {akruasi_train:.4f}")

NameError: name 'X_train' is not defined

### Prediksi Test

In [ ]:
prediksi_test = prediksi(X_test, w, b)
akruasi_test = np.sum(prediksi_train == y_test) / len(X_test)
print(f"Prediksi Train\nAkurasi Train: {akruasi_test:.4f}")

### Akurasi, Recall, Presisi, F1

In [ ]:
accuracy_train, precision_train, recall_train, f1_train = evaluate(y_train, prediksi_train)
accuracy_test, precision_test, recall_test, f1_test = evaluate(y_train, prediksi_test)

print(f"Train")
print(f"\n- Akurasi: {accuracy_train}\n- Presisi: {precision_train}\n- Recall: {recall_train}\n- F1: {f1_train}")
print(f"\nTest")
print(f"\n- Akurasi: {accuracy_test}\n- Presisi: {precision_test}\n- Recall: {recall_test}\n- F1: {f1_test}")

## **VISUALISASI**

### Sigmoid

In [ ]:
z_1 = linear(X_test, w, b)
z_sort = np.sort(z_1)
z_sorted = sigmoid(z_sort)

plt.plot(z_sort, z_sorted)
plt.grid(True)
plt.title("Visualisasi Sigmoid")
plt.show()

### Confusion Matrix

In [ ]:
def confusion_matrix(y_true, y_pred):
    labels = np.unique(np.concatenate((y_true - y_pred)))
    labels_idx = {label: i for i, label in enumerate(labels)}

    cm = np.zeros((len(labels), len(labels)), dtypes=int)
    for yt, yp in zip(y_true, y_pred):
        i = labels_idx[yt]
        j = labels_idx[yp]
        cm[i, j] += 1
    return cm, labels

cm, labels = confusion_matrix(y_test, prediksi_test)
sns.heatmap(cm, annot=True, cmap='Blue', fmt='d')
plt.show()